# Dört kat DNS veri seti ile NumPy GRU

Ömer Altın ve Muhammed Aslan — EE491–EE492

Bu paket **800 farklı okuyucu kimliğinden temiz kayıt, 80 gürültü kaynağı ve 2.400
temiz–gürültülü çift** içerir. Önceki 200/20/600 paketine göre toplam veri dört katıdır.
Yeni 600 temiz kayıt ve 60 gürültü kaynağının tamamı eğitime eklenmiştir.

| Grup | Temiz kayıt | Gürültü kaynağı | Gürültülü çift |
|---|---:|---:|---:|
| Train | 740 | 74 | 2.220 |
| Validation | 30 | 3 | 90 |
| Test | 30 | 3 | 90 |

Önceki tüm 600 çift ve 820 WAV byte düzeyinde korunur. Doğrulama/test değişmediği
için aynı seslerde karşılaştırma yapılabilir. Bu gruplar önceki geliştirmede görüldü;
yeni, kör benchmark değildir. Kaynak okuyucular ve gürültüler gruplar arasında ayrıdır.

**Bu defter yeni veride sıfırdan eğitim yapar.** Önceki paketin hazır ağırlıkları yüklenmez.
Model aynı 48-birimli NumPy GRU'dur; SNN veya PyTorch kullanılmaz. Daha çok verinin
kaliteye etkisi eğitimden sonra ölçülecek; bu veri paketinde yeni model henüz eğitilmedi.

Colab'a bu defteri yükleyin ve Drive'a kaydedin. İlk hücre isterse bu paketin tamamını
**DNS_MLP_4x.zip** olarak yükleyin. Eski DNS_MLP_Baslangic.zip dosyasını seçmeyin.

In [ ]:
from pathlib import Path
import csv, json, time, io, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import ShortTimeFFT

ROOT = Path('/content/DNS_MLP_4x')
if (Path.cwd() / 'dataset_summary.json').exists():
    candidate = json.loads((Path.cwd() / 'dataset_summary.json').read_text())
    if candidate.get('dataset_id') == 'DNS_MLP_4x_20261031':
        ROOT = Path.cwd()

if not (ROOT / 'pairs.csv').exists():
    from google.colab import files
    print('Dosyaları Seç düğmesine basıp YENİ DNS_MLP_4x.zip dosyasını seçin.')
    print('Bu hücre dosya seçilene ve yükleme bitene kadar bekler; henüz eğitim yapmaz.')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        assert 'DNS_MLP_4x/pairs.csv' in archive.namelist(), 'Yeni DNS_MLP_4x.zip dosyasını seçin.'
        archive.extractall('/content')
    ROOT = Path('/content/DNS_MLP_4x')
    del uploaded

summary = json.loads((ROOT / 'dataset_summary.json').read_text())
assert summary['dataset_id'] == 'DNS_MLP_4x_20261031'
with (ROOT / 'pairs.csv').open(encoding='utf-8', newline='') as f:
    pairs = list(csv.DictReader(f))
assert len(pairs) == 2400
for split in ['train', 'validation', 'test']:
    rows = [r for r in pairs if r['split'] == split]
    print(split, 'temiz kayıt:', len({r['clean_id'] for r in rows}),
          'gürültü kaynağı:', len({r['noise_id'] for r in rows}), 'çift:', len(rows))

OUTPUT_DIR = ROOT / 'numpy_gru_4x_results'
OUTPUT_DIR.mkdir(exist_ok=True)
transform = ShortTimeFFT(np.hanning(512), hop=128, fs=16000, fft_mode='onesided')
HIDDEN = 48
EPOCHS = 15
BATCH = 16
TBPTT = 64
LEARNING_RATE = 0.001
print('Yeni veri hazır:', ROOT)
print('GRU: 257 → 48 gizli birim → 257 maske; SNN kullanılmıyor.')

## 1. Veri gruplarını kontrol etme

Bu hücre okuyucu ve gürültü kaynaklarının gruplar arasında karışmadığını kontrol eder.
SNR manifest değerleri ve korunmuş önceki çift sayısı da kontrol edilir. Üretim sırasında
paketteki tüm WAV dosyalarının uzunlukları, biçimleri, SHA256'ları ve gerçek dalga biçimi
SNR değerleri ayrıca kontrol edilmiştir.

In [ ]:
for field in ['speaker_id', 'noise_id']:
    groups = {split: {row[field] for row in pairs if row['split'] == split}
              for split in ['train', 'validation', 'test']}
    for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
        assert not groups[a] & groups[b], f'{field} grupları karışmış!'
assert summary['validation_and_test_unchanged']
assert summary['original_pairs_preserved'] == 600
assert max(abs(float(row['measured_snr_db']) - float(row['target_snr_db'])) for row in pairs) < 0.02
print('Kaynak ayrımı, önceki verinin korunması ve manifest SNR kontrolleri geçti.')
print('Dikkat: doğrulama/test daha önce geliştirmede kullanıldı; yeni kör benchmark değil.')

## 2. Sesleri dizi biçiminde hazırlama

read_audio: 16 kHz mono PCM16 ses okuma. prepare: log-genlik girdisi, faza duyarlı
maske hedefi ve güç ağırlıklarını hazırlama. Train dizilerinin her biri (2220,628,257),
validation (90,628,257) olur. Aynı kaydın zaman sırası korunur.

Üç büyük train dizisi toplam yaklaşık 4.3 GB (ondalık) ham float32 bellek kullanır;
işlem sırasında başka geçici diziler de oluşur. Yeni bir Colab oturumunda çalıştırın.
Önceki deneylerin büyük dizilerini RAM'de tutmak bellek yetersizliğine yol açabilir.

In [ ]:
def read_audio(relative):
    fs,x=wavfile.read(ROOT/relative)
    assert fs==16000 and x.shape==(80000,) and x.dtype==np.int16
    return x.astype(np.float64)/32768


def prepare(split):
    rows=[r for r in pairs if r['split']==split]
    frames=transform.stft(read_audio(rows[0]['noisy_path'])).shape[1]
    X=np.empty((len(rows),frames,257),dtype=np.float32)
    T=np.empty_like(X);P=np.empty_like(X)
    for i,row in enumerate(rows):
        clean=transform.stft(read_audio(row['clean_path']))
        noisy=transform.stft(read_audio(row['noisy_path']))
        power=abs(noisy)**2
        X[i]=np.log1p(abs(noisy)).T
        T[i]=(np.real(clean*np.conj(noisy))/np.maximum(power,1e-8)).T
        P[i]=power.T
    print('Prepared',split,X.shape,flush=True)
    return X,T,P

## 3. GRU ve kayıp

Mimari önceki modelle aynı: 257 giriş, 48 gizli birim, 257 sigmoid maske; 56.657 parametre.
z bu kodda yeni adayın ağırlığıdır; r adayın geçmişten yararlanmasını kontrol eder.
h=(1-z)*previous+z*q ile gizli durum taşınır. Kayıp, güç ağırlıklı maske karesel hatasıdır.
Normalizasyon yalnız train'den öğrenilir. Yeni verinin etkisini incelerken mimariyi büyütmüyoruz.

In [ ]:
def sigmoid(x):
    return np.exp(-np.logaddexp(0,-x))


def initialize():
    rng=np.random.default_rng(42)
    p={}
    for gate in ['z','r','q']:
        p['W'+gate]=(rng.standard_normal((257,HIDDEN))/np.sqrt(257)).astype(np.float32)
        q,_=np.linalg.qr(rng.standard_normal((HIDDEN,HIDDEN)))
        p['U'+gate]=(q*.9).astype(np.float32)
        p['b'+gate]=np.zeros((1,HIDDEN),dtype=np.float32)
    p['Wo']=np.zeros((HIDDEN,257),dtype=np.float32)
    p['bo']=np.full((1,257),np.log(.9/.1),dtype=np.float32)
    return p


def forward_gru(X,p,h0=None,keep_cache=True):
    # X: (record batch, time, frequency). h0 carries previous CHUNK state.
    h=np.zeros((len(X),p['Uz'].shape[0]),dtype=X.dtype) if h0 is None else h0.copy()
    masks=np.empty_like(X)
    caches=[] if keep_cache else None
    for t in range(X.shape[1]):
        x=X[:,t,:];previous=h
        z=sigmoid(x@p['Wz']+previous@p['Uz']+p['bz'])
        r=sigmoid(x@p['Wr']+previous@p['Ur']+p['br'])
        q=np.tanh(x@p['Wq']+(r*previous)@p['Uq']+p['bq'])
        # Our z weights the CANDIDATE. Other conventions interchange z and 1-z.
        h=(1-z)*previous+z*q
        masks[:,t,:]=sigmoid(h@p['Wo']+p['bo'])
        if keep_cache:caches.append((x,previous,z,r,q,h))
    return masks,h,caches


def objective(mask,target,power):
    return float(np.sum(power*(mask-target)**2,dtype=np.float64)/(np.sum(power,dtype=np.float64)+1e-12))

## 4. Zaman boyunca geri yayılım ve Adam

backward_gru hata türevlerini zaman içinde sondan başa taşır. adam_step önce küresel
gradyan normunu en fazla 5 yapar, ardından Adam uygular. İleri/geri yayılım ve
optimizasyon NumPy ile yazılmıştır; hazır bir sinir ağı eğitim kütüphanesi kullanılmaz.

In [ ]:
def backward_gru(mask,target,power,caches,p):
    grads={k:np.zeros_like(v) for k,v in p.items()}
    dm=2*power*(mask-target)/(np.sum(power,dtype=np.float64)+1e-12)
    dh_next=np.zeros_like(caches[0][1])
    for t in reversed(range(mask.shape[1])):
        x,previous,z,r,q,h=caches[t]
        do=dm[:,t,:]*mask[:,t,:]*(1-mask[:,t,:])
        grads['Wo']+=h.T@do;grads['bo']+=do.sum(axis=0,keepdims=True)
        dh=do@p['Wo'].T+dh_next
        dq=dh*z;dz=dh*(q-previous);dh_previous=dh*(1-z)
        aq=dq*(1-q*q)
        grads['Wq']+=x.T@aq;grads['Uq']+=(r*previous).T@aq
        grads['bq']+=aq.sum(axis=0,keepdims=True)
        dproduct=aq@p['Uq'].T
        dr=dproduct*previous;dh_previous+=dproduct*r
        ar=dr*r*(1-r)
        grads['Wr']+=x.T@ar;grads['Ur']+=previous.T@ar
        grads['br']+=ar.sum(axis=0,keepdims=True)
        dh_previous+=ar@p['Ur'].T
        az=dz*z*(1-z)
        grads['Wz']+=x.T@az;grads['Uz']+=previous.T@az
        grads['bz']+=az.sum(axis=0,keepdims=True)
        dh_previous+=az@p['Uz'].T
        dh_next=dh_previous
    return grads


def adam_step(p,grads,moments,velocities,step):
    # Global norm clipping is applied BEFORE the Adam update.
    norm=np.sqrt(sum(float(np.sum(g.astype(np.float64)**2)) for g in grads.values()))
    scale=min(1.,5./max(norm,1e-12))
    for key in p:
        g=grads[key]*scale
        moments[key]*=.9;moments[key]+=.1*g
        velocities[key]*=.999;velocities[key]+=.001*g*g
        corrected_m=moments[key]/(1-.9**step)
        corrected_v=velocities[key]/(1-.999**step)
        p[key]-=LEARNING_RATE*corrected_m/(np.sqrt(corrected_v)+1e-8)

## 5. Eğitim fonksiyonlarını tanımlama

15 epoch, 16 kayıt/batch, 64 adımlık truncated BPTT, learning rate .001 ve aynı tohumlar
korunur. h aynı kaydın sonraki parçasına taşınır; yeni kayıtta sıfırlanır. 64 adımda
bir türev zinciri kesilir, gizli durum değeri korunur. En yüksek ortalama validation
dalga biçimi SNR'sini veren checkpoint kaydedilir.

Bu hücre fonksiyonları tanımlar; eğitim sonraki kod hücresinde başlar.
Train loss epoch boyunca değişen ağırlıklarla hesaplanan online kayıptır; validation
loss epoch sonunda sabit ağırlıklarla hesaplanır. SNR ve maske kaybı farklı ölçütlerdir.

In [ ]:
def predict_sequences(X,p):
    # Each sample is an entire separate file; h is zeroed for every batch.
    predictions=np.empty_like(X)
    for start in range(0,len(X),BATCH):
        predictions[start:start+BATCH]=forward_gru(X[start:start+BATCH],p,keep_cache=False)[0]
    return predictions


def snr(reference,estimate):
    return float(10*np.log10((np.sum(reference**2)+1e-12)/(np.sum((reference-estimate)**2)+1e-12)))


def validation_audio_scores(masks):
    rows=[r for r in pairs if r['split']=='validation']
    values=[]
    for mask,row in zip(masks,rows):
        clean=read_audio(row['clean_path']);noisy=read_audio(row['noisy_path'])
        output=transform.istft(mask.T*transform.stft(noisy),k1=len(noisy))
        values.append(snr(clean,output))
    return float(np.mean(values))


def run_training():
    X,T,P=prepare('train');V,U,Q=prepare('validation')
    # Accumulate over records, avoiding full float64 copies of large arrays.
    mean=np.mean(X,axis=(0,1),keepdims=True,dtype=np.float64)
    variance=np.zeros_like(mean)
    for record in X:
        d=record.astype(np.float64)-mean[0]
        variance+=np.sum(d*d,axis=0,keepdims=True)[None,:,:]
    std=np.maximum(np.sqrt(variance/(X.shape[0]*X.shape[1])),1e-4).astype(np.float32)
    mean=mean.astype(np.float32)
    X-=mean;X/=std;V-=mean;V/=std
    p=initialize()
    moments={k:np.zeros_like(v) for k,v in p.items()};velocities={k:np.zeros_like(v) for k,v in p.items()}
    best={k:v.copy() for k,v in p.items()}
    initial=predict_sequences(V,p)
    best_snr=validation_audio_scores(initial);best_loss=objective(initial,U,Q);best_epoch=0
    rng=np.random.default_rng(123);history=[];step=0
    for epoch in range(1,EPOCHS+1):
        tic=time.perf_counter();order=rng.permutation(len(X));numerator=denominator=0.
        for start in range(0,len(X),BATCH):
            ii=order[start:start+BATCH]
            # Slots keep the same recording for all chunks in this batch.
            h=None
            for t in range(0,X.shape[1],TBPTT):
                xb=X[ii,t:t+TBPTT,:];tb=T[ii,t:t+TBPTT,:];pb=P[ii,t:t+TBPTT,:]
                masks,h,caches=forward_gru(xb,p,h0=h)
                numerator+=float(np.sum(pb*(masks-tb)**2,dtype=np.float64))
                denominator+=float(np.sum(pb,dtype=np.float64))
                grads=backward_gru(masks,tb,pb,caches,p)
                step+=1;adam_step(p,grads,moments,velocities,step)
                # h is carried numerically; derivatives stop at each chunk edge.
        predicted=predict_sequences(V,p)
        val=objective(predicted,U,Q);val_snr=validation_audio_scores(predicted)
        history.append({'epoch':epoch,'online_train_loss':numerator/denominator,
                        'validation_loss':val,'validation_snr':val_snr,
                        'seconds':time.perf_counter()-tic})
        if val_snr>best_snr:
            best_snr=val_snr;best_loss=val;best_epoch=epoch;best={k:v.copy() for k,v in p.items()}
        print('EPOCH',epoch,'online_train',round(numerator/denominator,6),'val_loss',round(val,6),
              'val_snr',round(val_snr,4),'seconds',round(time.perf_counter()-tic,2),flush=True)
    np.savez(OUTPUT_DIR/'gru_weights.npz',**best,x_mean=mean,x_std=std,
        hidden=np.array(HIDDEN),fs=np.array(16000),window=np.array(512),hop=np.array(128),
        best_epoch=np.array(best_epoch),best_validation_snr=np.array(best_snr),
        best_validation_loss=np.array(best_loss),tbptt=np.array(TBPTT))
    (OUTPUT_DIR/'gru_training_history.json').write_text(json.dumps(history,indent=2))
    print('BEST_EPOCH',best_epoch,'VALIDATION_SNR',best_snr,flush=True)
    return best,mean,std

## 6. Yeni veriyle sıfırdan eğitim

Bu hücreyi çalıştırınca eski checkpoint yüklenmez; yeni ağ başlatılır. Eğitimde
2.220/420 ≈ 5.29 kat çift bulunduğu için epoch başına daha fazla güncelleme yapılır.
Aynı 15 epoch ayarında toplam eğitim işi de artar. Sonuç farkı yalnız çeşitlilik
etkisini izole etmez; güncelleme sayısı da değişmiştir.

Veri yükleme süresi eğitimden ayrıdır. Özellik hazırlığından sonra her epoch'un
süresi ekranda yazar. İlk epoch süresinin yaklaşık 15 katı toplam eğitim süresi için
başlangıç tahmini verir. NumPy CPU kullanır; GPU seçmek bu kodu GPU'ya taşımaz.

In [ ]:
# Yeni veriyle sıfırdan eğitim: eski 200 kayıtlık çalışmanın ağırlıkları yüklenmez.
# Bu hücreyi tekrar çalıştırmak yeni bir eğitim başlatır ve bu klasördeki sonucu günceller.
print('EĞİTİM BAŞLADI — ilk epoch öncesinde özellikler hazırlanıyor.')
print('Önceki küçük çalışmaya göre epoch başına yaklaşık 5.3 kat eğitim verisi var.')
gru4x_params, gru4x_mean, gru4x_std = run_training()

history = json.loads((OUTPUT_DIR / 'gru_training_history.json').read_text())
with np.load(OUTPUT_DIR / 'gru_weights.npz', allow_pickle=False) as checkpoint:
    print('Seçilen epoch:', int(checkpoint['best_epoch']))
    print('Doğrulama ortalama SNR:', float(checkpoint['best_validation_snr']))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
epochs = [h['epoch'] for h in history]
axes[0].plot(epochs, [h['online_train_loss'] for h in history], label='Online training')
axes[0].plot(epochs, [h['validation_loss'] for h in history], label='Validation')
axes[0].set(xlabel='Epoch', ylabel='Power-weighted mask MSE')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [h['validation_snr'] for h in history])
axes[1].axhline(6.801504415565154, ls='--', color='gray', label='Previous small-data GRU')
axes[1].set(xlabel='Epoch', ylabel='Mean validation waveform SNR (dB)')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.savefig(OUTPUT_DIR / 'gru_training_curve.png', dpi=150)
plt.show()

## 7. Aynı doğrulama ve test seslerinde değerlendirme

enhance yalnız gürültülü sesi alır, sigmoid maskeyi gürültülü kompleks STFT'ye
uygular ve ters STFT ile ses üretir. Temiz referans sadece ölçümde kullanılır.
Gürültülü faz korunur, maske 0–1 arasındadır. SNR iyileşmesi tam gürültü temizliği
veya algısal kalite garantisi değildir. Gerçek zamanlı/FPGA çalışma ölçülmez.

In [ ]:
def enhance(noisy,p,mean,std):
    spectrum=transform.stft(noisy)
    X=np.log1p(abs(spectrum)).T.astype(np.float32)[None,:,:]
    masks=forward_gru((X-mean)/std,p,keep_cache=False)[0][0]
    output=transform.istft(masks.T*spectrum,k1=len(noisy))
    assert np.isfinite(output).all() and masks.min()>=0 and masks.max()<=1
    return output


def score(split,p,mean,std):
    results=[]
    for row in pairs:
        if row['split']!=split:continue
        clean=read_audio(row['clean_path']);noisy=read_audio(row['noisy_path'])
        output=enhance(noisy,p,mean,std)
        before=snr(clean,noisy);after=snr(clean,output)
        results.append({'clean_id':row['clean_id'],'snr':int(row['target_snr_db']),
                        'before':before,'after':after,'improvement':after-before})
    for level in [0,5,10]:
        rows=[r for r in results if r['snr']==level]
        print(split,level,{k:float(np.mean([r[k] for r in rows])) for k in ['before','after','improvement']},flush=True)
    return results



validation_results = score('validation', gru4x_params, gru4x_mean, gru4x_std)
test_results = score('test', gru4x_params, gru4x_mean, gru4x_std)
for split, rows in [('validation', validation_results), ('test', test_results)]:
    with (OUTPUT_DIR / f'{split}_snr.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)

# Önceki 48-birimli GRU ile karşılaştırma: aynı test sesleri.
old_test = {0: 1.9540367722225431, 5: 6.631705836303851, 10: 11.011009771557706}
print('Giriş SNR | Önceki GRU çıkışı | Yeni GRU çıkışı | Fark')
comparison = []
for level in [0, 5, 10]:
    values = [r['after'] for r in test_results if r['snr'] == level]
    new = float(np.mean(values))
    print(f'{level:2d} dB | {old_test[level]:.3f} dB | {new:.3f} dB | {new-old_test[level]:+.3f} dB')
    comparison.append({'input_snr_db': level, 'old_gru_output_snr_db': old_test[level],
                       'new_gru_output_snr_db': new, 'change_db': new-old_test[level]})
with (OUTPUT_DIR / 'comparison_with_small_dataset.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(comparison[0].keys()))
    writer.writeheader(); writer.writerows(comparison)

(OUTPUT_DIR / 'experiment_settings.json').write_text(json.dumps({
    'dataset_id': summary['dataset_id'], 'training_pairs': 2220,
    'validation_pairs': 90, 'test_pairs': 90, 'hidden': HIDDEN,
    'epochs': EPOCHS, 'batch_records': BATCH, 'tbptt': TBPTT,
    'learning_rate': LEARNING_RATE, 'normalization_fit': 'train only',
    'checkpoint_selection': 'highest mean validation waveform SNR',
    'old_validation_and_test_preserved': True,
    'test_status': 'previously seen development comparison'
}, indent=2))

## 8. Öncekiyle aynı örneği dinleme

clean_000, 5 dB örneği yine sabit seçilir. Üç kayda aynı oynatma kazancı uygulanır.
Yeni sonuç kötü çıkarsa da kaydedin; en iyi duyulan örneği seçerek raporlamayın.

In [ ]:
from IPython.display import Audio, display
row = next(r for r in pairs if r['split'] == 'test' and r['target_snr_db'] == '5')
reference = read_audio(row['clean_path'])
noisy = read_audio(row['noisy_path'])
output = enhance(noisy, gru4x_params, gru4x_mean, gru4x_std)
print('Aynı sabit test örneği:', row['clean_id'])
print('Giriş SNR:', snr(reference, noisy))
print('Yeni çıkış SNR:', snr(reference, output))
signals = [('clean_reference', reference), ('noisy_input', noisy), ('gru_4x_output', output)]
gain = 0.95 / max(1.0, *[float(np.max(np.abs(x))) for _, x in signals])
for label, signal in signals:
    print(label)
    playback = signal * gain
    display(Audio(playback, rate=16000, normalize=False))
    wavfile.write(OUTPUT_DIR / f'{label}.wav', 16000,
                 np.rint(playback * 32767).astype(np.int16))

## 9. Sonuçları indirme

Yalnız deney çıktıları indirilir; büyük veri seti tekrar paketlenmez. Yeni eğitim
geçmişi, ağırlıklar, grafik, CSV ölçümleri ve örnek WAV'lar sonuç ZIP'inde bulunur.

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/GRU_4x_Sonuclarim', 'zip', OUTPUT_DIR)
files.download(archive)

## Kaynak ve derslerle ilişkisi

Course 1: ileri/geri yayılım ve gradyanla öğrenme. Course 2: normalizasyon, Adam ve
doğrulama ile seçim. Course 5: diziler, GRU kapıları, gizli durum ve zaman boyunca geri yayılım.
Bu deney nihai Spiking-FullSubNet uygulaması değildir.

- DNS3: https://github.com/microsoft/DNS-Challenge/blob/master/README-DNS3.md
- Cho et al., GRU: https://arxiv.org/html/1406.1078v3#S2.SS3
- Goodfellow et al., diziler: https://www.deeplearningbook.org/contents/rnn.html
- Her kaynağın arşiv üyesi, grup kimliği ve SHA256'sı sources.csv içindedir.
- Kaynak lisansları bu paketle değişmez; SOURCE_DNS3_README.md dosyasına bakın.